# Scraping Data Ulasan Play Store (Indonesia)

Notebook ini mengambil data content aplikasi Duolingo secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store

In [1]:
%pip install -q google-play-scraper pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "id.dana"
APP_NAME = "DANA"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 10000
MAX_BATCH = 200

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = "dataset_dana.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.MOST_RELEVANT,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                }
            )

        if token is None:
            break

    return all_rows

In [ ]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal content terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total content terkumpul (raw): 10.000


,user_name,score,content
0,Irham Alfurqon,5,"mantap PLN, saya bisa lapor keluhan dan respon petugas gercep sekaligus handal"
1,IRFAN EKSEL,5,sangat senang menggunakan aplikasi dana apalagi udah pakai vitur paylater
2,Kurnia Kur,1,duit gua ilang ka tangung jawab
3,Sakura Rara,5,Kren
4,Roreng Belk,4,lumayan


In [ ]:
df = raw_df.copy()

print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "user_name", "score", "content"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Distribusi skor:


score
1    1927
2     406
3     433
4     650
5    6584
Name: count, dtype: int64


File CSV tersimpan di: dataset_dana.csv
Jumlah sampel akhir: 10.000


In [ ]:
df.sample(5, random_state=42)[["user_name", "score", "content"]]

,user_name,score,content
6252,pingki purdianti,5,good
4684,Marlon Albert,5,mantap
1731,Joni zebua,1,"setelah update, perangkat dana saya banyak mslh, dan sampe skrg saya blm dpt solusi dr aplikasi diana"
4742,Sut Cica,5,kren
4521,Hafizh Zulkarnain,1,"sering terjadi permasalahan pada sistem pembayaran menggunakan kode QR, sering juga mengalami proses transaksi yg sa..."
